#### Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)
    
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [3]:
# First Question:
len(documents)

72

In [4]:
from rag import build_index, search

query = "How does the agentic loop keep calling the model until it stops?"

minsearch_index = build_index(documents)
search_results = search(query, minsearch_index)

In [5]:
# Second Question:
search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [6]:
from rag import ask_assistant
# Third Question:
query = "How does the agentic loop keep calling the model until it stops?"
answer, token = ask_assistant(query, minsearch_index)

print("Answer:", answer)
print("-------------")
print("Prompt Token Usage:", token.prompt_tokens)
print("Completion Token Usage:", token.completion_tokens)
print("Total Token Usage:", token.total_tokens)

Answer: The agentic loop keeps calling the model until it stops by repeating the following steps:

1. Send a message to the user.
2. Run any function calls (tools) that are part of the agent's instruction set.
3. Repeat step 2 until the model is satisfied and decides to stop.

This process continues until the model has produced an answer or has decided to terminate the conversation. The loop then repeats with a new message from the user, running tools again as necessary until the model stops.
-------------
Prompt Token Usage: 4096
Completion Token Usage: 102
Total Token Usage: 4198


In [7]:
# Q4: Chunking
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [8]:
# Q5: RAG with chunking
minsearch_index = build_index(chunks)

query = "How does the agentic loop keep calling the model until it stops?"
answer, token = ask_assistant(query, minsearch_index)

print("Answer:", answer)
print("-------------")
print("Prompt Token Usage:", token.prompt_tokens)
print("Completion Token Usage:", token.completion_tokens)
print("Total Token Usage:", token.total_tokens)

Answer: It keeps calling the model until it returns a response without any function calls.
-------------
Prompt Token Usage: 2398
Completion Token Usage: 16
Total Token Usage: 2414


In [9]:
# Q6: Turning it into an agent
from openai import OpenAI
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback
from toyaikit.llm import OpenAIChatCompletionsClient

In [10]:
# Suppress the unknown model warning for local llama3.2 cost calculation
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [11]:
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

In [12]:
llm_client = OpenAIChatCompletionsClient(
    client=openai_client,
    model="llama3.2"
)

In [13]:
# Define the search tool (using your actual minsearch_index variable)
def search(query: str) -> str:
    """
    Search the FAQ database for relevant documents based on the query and return the top result.
    """
    results = minsearch_index.search(query, num_results=3)
    
    context_str = ""
    for doc in results:
        context_str += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"
    return context_str

agent_tools = Tools()
agent_tools.add_tool(search)

In [14]:
# Define the instructions required by the assignment
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

# Set up chat interface, callback, and the correct chat-completions runner
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [15]:
# Run the agent with the Q6 question
question = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


In [16]:
from agent import agent_loop
# 1. Define the system instructions required by the assignment
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

# 2. Define the exact question from Q6
question = "How does the agentic loop work, and how is it different from plain RAG?"

print("\n🚀 Starting the Agentic Loop...\n")

# 3. Execute your agent loop using your minsearch_index
answer, total_searches = agent_loop(
    instructions=instructions, 
    question=question, 
    index_client=minsearch_index
)

# 4. Print the final results and tool call count
print("\n====================================")
print("FINAL ANSWER:\n", answer)
print("====================================")
print(f"Total Search Tool Calls: {total_searches}")


🚀 Starting the Agentic Loop...


[Iteration #1] Model is thinking...
 -> Caught text-based tool call from Llama 3.2.
 -> Sanitized query sent to database: 'agentic loop vs plain RAG'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

FINAL ANSWER:
 The agentic loop is a pattern used in AI agents to manage the conversation flow. It consists of a while loop that calls the LLM, executes any tool calls it returned, sends the results back, and stops when the model produced a final answer with no more tool calls.

In contrast, plain RAG (Reactive Agent Framework) uses a traditional workflow approach, where the sequence of steps is predetermined and fixed. The agent follows a predefined logic to achieve its goal.

The main difference between the agentic loop and plain RAG is that the agentic loop is dynamic and adaptive, allowing the agent to make decisions based on changing information or unexpected conditions. In contrast, plain RAG uses a more rigid approach, where the sequ

In [17]:
from agent import agent_loop

In [19]:
instructions = """
You are an advanced investigative assistant.
Your goal is to answer the user's question completely based on search results.
Make multiple searches with different keywords before answering.
""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"

print("Starting the Agentic Loop...")
final_answer, total_searches = agent_loop(instructions, question, minsearch_index)

print("\n====================================")
print("FINAL ANSWER:\n", final_answer)
print("====================================")
print(f"Total Search Tool Calls: {total_searches}")

Starting the Agentic Loop...

[Iteration #1] Model is thinking...
 -> Caught text-based tool call from Llama 3.2.
 -> Sanitized query sent to database: 'agentic loop vs plain RAG'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

FINAL ANSWER:
 The agentic loop is a pattern used in AI agents to manage the conversation flow. It consists of a while loop that calls the LLM, executes any tool calls it returned, sends the results back, and stops when the model produced a final answer with no more tool calls.

In contrast, plain RAG (Reactive Agent Framework) uses a traditional workflow approach, where the sequence of steps is predetermined and fixed. The agent follows a predefined logic to achieve its goal.

The main difference between the agentic loop and plain RAG is that the agentic loop is more flexible and adaptive, allowing the agent to decide what to do in each step based on the current situation. This makes it suitable for situations where the exact sequence of steps